# Multiclass Classification of Car Acceptability
### Decision Tree vs. XGBoost on the UCI Car Evaluation Dataset

## Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo
SEED = 445

## Load Dataset & Splitting 
As per the assignment instruction keeping : the **top 70%** of rows for training, the **next 15%**
for validation, and the **last 15%** for testing
The target `class` is mapped to integers using the natural acceptability
ordering (`unacc`=0 < `acc`=1 < `good`=2 < `vgood`=3) so that the confusion
matrix rows/columns have an interpretable order.

In [2]:
car_evaluation = fetch_ucirepo(id=19)
X = car_evaluation.data.features
y = car_evaluation.data.targets

class_map = {"unacc": 0, "acc": 1, "good": 2, "vgood": 3}

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, shuffle=False)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

print("\nClass distribution — Train:", y_train.value_counts().sort_index().to_dict())
print("Class distribution — Validation:", y_val.value_counts().sort_index().to_dict())
print("Class distribution — Test:", y_test.value_counts().sort_index().to_dict())

Train size: 1209
Validation size: 259
Test size: 260

Class distribution — Train: {('acc',): 287, ('good',): 3, ('unacc',): 905, ('vgood',): 14}
Class distribution — Validation: {('acc',): 61, ('good',): 20, ('unacc',): 161, ('vgood',): 17}
Class distribution — Test: {('acc',): 36, ('good',): 46, ('unacc',): 144, ('vgood',): 34}


## Feature Encoding using one hot encoder 

In [3]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.transform(X_val)
X_test_encoded = encoder.transform(X_test)

print("Encoded feature matrix shape (train):", X_train_encoded.shape)

Encoded feature matrix shape (train): (1209, 14)


e:\CSE445\CSE445\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
e:\CSE445\CSE445\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Custom Evaluation Metrics
We manually implement 
Confusion Matrix, Accuracy, and **macro-averaged** Precision / Recall /
F-score, with NumPy.

In [4]:
def custom_confusion_matrix(y_actual, y_predicted):
    y_true = np.array(y_actual, dtype=int)
    y_pred = np.array(y_predicted, dtype=int)

    num_classes = max(np.max(y_true), np.max(y_pred)) + 1
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        confusion_matrix[actual][predicted] += 1

    return confusion_matrix


def custom_accuracy(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)
    correct = np.trace(confusion_matrix)
    total = np.sum(confusion_matrix)
    return correct / total if total > 0 else 0.0


def custom_precision(y_actual, y_predicted):
    """Macro-averaged precision: mean of per-class TP / (TP + FP)."""
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)
    precision_scores = []
    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseP = np.sum(confusion_matrix[:, i]) - trueP
        precision_scores.append(0.0 if trueP + falseP == 0 else trueP / (trueP + falseP))
    return np.mean(precision_scores)


def custom_recall(y_actual, y_predicted):
    """Macro-averaged recall: mean of per-class TP / (TP + FN)."""
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)
    recall_scores = []
    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseN = np.sum(confusion_matrix[i, :]) - trueP
        recall_scores.append(0.0 if trueP + falseN == 0 else trueP / (trueP + falseN))
    return np.mean(recall_scores)


def custom_f1(y_actual, y_predicted):

    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)
    f1_scores = []
    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseP = np.sum(confusion_matrix[:, i]) - trueP
        falseN = np.sum(confusion_matrix[i, :]) - trueP
        precision = 0.0 if trueP + falseP == 0 else trueP / (trueP + falseP)
        recall = 0.0 if trueP + falseN == 0 else trueP / (trueP + falseN)
        f1_scores.append(0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall))
    return np.mean(f1_scores)


def custom_classification_report(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)
    return {
        "accuracy": custom_accuracy(y_actual, y_predicted),
        "avg_precision": custom_precision(y_actual, y_predicted),
        "avg_recall": custom_recall(y_actual, y_predicted),
        "avg_f1": custom_f1(y_actual, y_predicted),
        "confusion_matrix": confusion_matrix,
    }

---
## Decision Tree Classifier


### We use hyperparameter tuning and choose our own

In [ ]:
DT_Hyperparameters = {
    "max_depth": [3, 5, 7, 10, 12, None],
    "min_samples_split": [2, 5, 10, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "criterion": ["gini", "entropy", "log_loss"],
}
results = []
best_accuracy = -1
best_performing_model = None
best_hyperparameters = None

for depth in DT_Hyperparameters["max_depth"]:
    for min_samples_split in DT_Hyperparameters["min_samples_split"]:
        for min_samples_leaf in DT_Hyperparameters["min_samples_leaf"]:
            for criterion in DT_Hyperparameters["criterion"]:
                model = DecisionTreeClassifier(
                    max_depth=depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=SEED
                )
                model.fit(X_train_encoded, y_train)
                y_pred = model.predict(X_val_encoded)
                accuracy = custom_accuracy(y_val, y_pred)
                results.append((depth, min_samples_split, min_samples_leaf, criterion, accuracy))
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_performing_model = model
                    best_hyperparameters = (depth, min_samples_split, min_samples_leaf, criterion)

dt_results_df = (pd.DataFrame(results, columns=["max_depth", "min_samples_split", "min_samples_leaf", "criterion", "val_accuracy"])
                  .sort_values(by="val_accuracy", ascending=False)
                  .reset_index(drop=True)
                  )
print("Best Hyperparameters (max_depth, min_samples_split, min_samples_leaf, criterion):", best_hyperparameters)
print("Best Validation Accuracy:", best_accuracy)
print("\nTop 10 combinations:")
print(dt_results_df.head(10))

Best Hyperparameters (max_depth, min_samples_split, min_samples_leaf, criterion): (3, 2, 1, 'gini')
Best Validation Accuracy: 0.7567567567567568

Top 10 combinations:
   max_depth  min_samples_split  min_samples_leaf criterion  val_accuracy
0        3.0                 30                 1      gini      0.756757
1        3.0                 20                20      gini      0.756757
2        3.0                 20                 2      gini      0.756757
3        3.0                 30                 2      gini      0.756757
4        3.0                 20                10      gini      0.756757
5        3.0                 20                 5      gini      0.756757
6        3.0                 10                20      gini      0.756757
7        3.0                 20                 1      gini      0.756757
8        3.0                 30                10      gini      0.756757
9        3.0                 30                 5      gini      0.756757


### Checking for Overfitting 

In [ ]:
header = f"{'max_depth':>10} | {'train_accuracy':>15} | {'val_accuracy':>13}"
print(header)
for depth in [best_hyperparameters[0], 15, 20, None]:
    probe_model = DecisionTreeClassifier(
        max_depth=depth, min_samples_split=2, min_samples_leaf=1,
        criterion="gini", random_state=SEED
    )
    probe_model.fit(X_train_encoded, y_train)
    train_acc = custom_accuracy(y_train, probe_model.predict(X_train_encoded))
    val_acc = custom_accuracy(y_val, probe_model.predict(X_val_encoded))
    print(f"{str(depth):>10} | {train_acc:>15.4f} | {val_acc:>13.4f}")

 max_depth |  train_accuracy |  val_accuracy
         3 |          0.8238 |        0.7568
        15 |          1.0000 |        0.7143
        20 |          1.0000 |        0.7143
      None |          1.0000 |        0.7143


As the table shows, letting the tree grow deeper takes the training
accuracy to 1.0 (perfect memorization of the training rows) while
*validation* accuracy actually **decreases** relative to the shallow,
depth-3 tree selected by the grid search. It kinda shows the bias–variance
trade-off: on this particular sequential split, a small, heavily regularized
tree generalizes better than a deep one.

### Decision Tree — Final Test Set Evaluation
The best model found on the validation set (Trial 2) is evaluated once on
the previously untouched test set.

In [ ]:
dt_model = best_performing_model
dt_val_report = custom_classification_report(y_val, dt_model.predict(X_val_encoded))
dt_test_pred = dt_model.predict(X_test_encoded)
dt_test_report = custom_classification_report(y_test, dt_test_pred)

print("Decision Tree — Best Hyperparameters:", best_hyperparameters)
print("Decision Tree — Validation Report:", dt_val_report)
print("Decision Tree — Test Report:", dt_test_report)

Decision Tree — Best Hyperparameters: (3, 2, 1, 'gini')
Decision Tree — Validation Report: {'accuracy': np.float64(0.7567567567567568), 'avg_precision': np.float64(0.3458433014354067), 'avg_recall': np.float64(0.4138071479482741), 'avg_f1': np.float64(0.37471698876041076), 'confusion_matrix': array([[153,   8,   0,   0],
       [ 18,  43,   0,   0],
       [  0,  20,   0,   0],
       [  0,  17,   0,   0]])}
Decision Tree — Test Report: {'accuracy': np.float64(0.676923076923077), 'avg_precision': np.float64(0.325), 'avg_recall': np.float64(0.4930555555555556), 'avg_f1': np.float64(0.361863488624052), 'confusion_matrix': array([[140,   4,   0,   0],
       [  0,  36,   0,   0],
       [  0,  46,   0,   0],
       [  0,  34,   0,   0]])}


### Making the Confusion Matrix for Test Set

In [ ]:
class_labels = ["unacc", "acc", "good", "vgood"]
dt_confusion_df = pd.DataFrame(
    dt_test_report["confusion_matrix"],
    index=[f"Actual {c}" for c in class_labels],
    columns=[f"Predicted {c}" for c in class_labels]
)
print("Decision Tree Confusion Matrix (Test Set):")
dt_confusion_df

Decision Tree Confusion Matrix (Test Set):


,Predicted unacc,Predicted acc,Predicted good,Predicted vgood
Actual unacc,140,4,0,0
Actual acc,0,36,0,0
Actual good,0,46,0,0
Actual vgood,0,34,0,0


---
## XGBoost Classifier

### Like the previous one we first figure out the Baseline performance with default hyperparameters

In [ ]:
baseline_xgb = XGBClassifier(
    objective="multi:softmax", num_class=4, random_state=SEED,
    n_jobs=4, verbosity=0, eval_metric="mlogloss"
)
baseline_xgb.fit(X_train_encoded, y_train)
baseline_xgb_val_acc = custom_accuracy(y_val, baseline_xgb.predict(X_val_encoded))
print("Baseline XGBoost — Validation Accuracy:", baseline_xgb_val_acc)

Baseline XGBoost — Validation Accuracy: 0.6872586872586872


### Using Fixed-Seed Random Search on Validation Set with our hyperparameters

In [ ]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}
results = []

best_accuracy = -1
best_model = None
best_params = None

rng = np.random.RandomState(SEED)

for _ in range(60):
    max_depth = rng.choice(XGB_GRID["max_depth"])
    learning_rate = rng.choice(XGB_GRID["learning_rate"])
    n_estimators = rng.choice(XGB_GRID["n_estimators"])
    subsample = rng.choice(XGB_GRID["subsample"])
    colsample_bytree = rng.choice(XGB_GRID["colsample_bytree"])
    min_child_weight = rng.choice(XGB_GRID["min_child_weight"])
    gamma = rng.choice(XGB_GRID["gamma"])

    model = XGBClassifier(
        objective="multi:softmax",
        num_class=4,
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
        eval_metric="mlogloss",
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma
    )

    model.fit(X_train_encoded, y_train)
    predictions = model.predict(X_val_encoded)
    accuracy = custom_accuracy(y_val, predictions)

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "gamma": gamma,
        "val_accuracy": accuracy
    })
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        model2 = model
        best_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma
        }

xgb_results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)

print("Best Hyperparameters:", best_params)
print("Best Validation Accuracy:", best_accuracy)
print("\nTop 10 combinations:")
print(xgb_results_df.head(10))

Best Hyperparameters: {'max_depth': np.int64(5), 'learning_rate': np.float64(0.1), 'n_estimators': np.int64(300), 'subsample': np.float64(0.8), 'colsample_bytree': np.float64(1.0), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.0)}
Best Validation Accuracy: 0.6911196911196911

Top 10 combinations:
   max_depth  learning_rate  n_estimators  subsample  colsample_bytree  \
0          5           0.10           300        0.8               1.0   
1          7           0.10           300        1.0               1.0   
2          3           0.01           300        1.0               1.0   
3          3           0.05           100        0.8               0.8   
4          3           0.01           300        0.8               1.0   
5          3           0.10           300        1.0               0.8   
6          3           0.05           100        0.8               1.0   
7          7           0.10           300        1.0               1.0   
8          5           0.01

### Performance Comparison Against the Decision Tree


### XGBoost: Final Test Set Evaluation

In [ ]:
xgb_model = model2
xgb_val_report = custom_classification_report(y_val, xgb_model.predict(X_val_encoded))
xgb_test_pred = xgb_model.predict(X_test_encoded)
xgb_test_report = custom_classification_report(y_test, xgb_test_pred)

print("XGBoost — Best Hyperparameters:", best_params)
print("XGBoost — Validation Report:", xgb_val_report)
print("XGBoost — Test Report:", xgb_test_report)

XGBoost — Best Hyperparameters: {'max_depth': np.int64(5), 'learning_rate': np.float64(0.1), 'n_estimators': np.int64(300), 'subsample': np.float64(0.8), 'colsample_bytree': np.float64(1.0), 'min_child_weight': np.int64(1), 'gamma': np.float64(0.0)}
XGBoost — Validation Report: {'accuracy': np.float64(0.6911196911196911), 'avg_precision': np.float64(0.525388349514563), 'avg_recall': np.float64(0.34498553519768566), 'avg_f1': np.float64(0.344049700065762), 'confusion_matrix': array([[161,   0,   0,   0],
       [ 45,  16,   0,   0],
       [  0,  20,   0,   0],
       [  0,  14,   1,   2]])}
XGBoost — Test Report: {'accuracy': np.float64(0.6), 'avg_precision': np.float64(0.2468944099378882), 'avg_recall': np.float64(0.3333333333333333), 'avg_f1': np.float64(0.2776442307692308), 'confusion_matrix': array([[144,   0,   0,   0],
       [ 24,  12,   0,   0],
       [  0,  46,   0,   0],
       [  0,  34,   0,   0]])}


### Confusion Matrix on Test Set

In [ ]:
xgb_confusion_df = pd.DataFrame(
    xgb_test_report["confusion_matrix"],
    index=[f"Actual {c}" for c in class_labels],
    columns=[f"Predicted {c}" for c in class_labels]
)
print("XGBoost Confusion Matrix (Test Set):")
xgb_confusion_df

XGBoost Confusion Matrix (Test Set):


,Predicted unacc,Predicted acc,Predicted good,Predicted vgood
Actual unacc,144,0,0,0
Actual acc,24,12,0,0
Actual good,0,46,0,0
Actual vgood,0,34,0,0


---
## Results Comparison
We compare the Validation and test performance for both models: Accuracy,
Precision, Recall, and  F-score and we use the custom functions for the comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Decision Tree", "Split": "Validation",
        "Accuracy": dt_val_report["accuracy"], "Avg Precision": dt_val_report["avg_precision"],
        "Avg Recall": dt_val_report["avg_recall"], "Avg F-score": dt_val_report["avg_f1"],
    },
    {
        "Model": "Decision Tree", "Split": "Test",
        "Accuracy": dt_test_report["accuracy"], "Avg Precision": dt_test_report["avg_precision"],
        "Avg Recall": dt_test_report["avg_recall"], "Avg F-score": dt_test_report["avg_f1"],
    },
    {
        "Model": "XGBoost", "Split": "Validation",
        "Accuracy": xgb_val_report["accuracy"], "Avg Precision": xgb_val_report["avg_precision"],
        "Avg Recall": xgb_val_report["avg_recall"], "Avg F-score": xgb_val_report["avg_f1"],
    },
    {
        "Model": "XGBoost", "Split": "Test",
        "Accuracy": xgb_test_report["accuracy"], "Avg Precision": xgb_test_report["avg_precision"],
        "Avg Recall": xgb_test_report["avg_recall"], "Avg F-score": xgb_test_report["avg_f1"],
    },
])

print("Decision Tree Confusion Matrix (Test):")
print(dt_confusion_df)
print()
print("XGBoost Confusion Matrix (Test):")
print(xgb_confusion_df)
print()
print("Final Comparison — Decision Tree vs. XGBoost (Validation & Test):")
comparison_df

Decision Tree Confusion Matrix (Test):
              Predicted unacc  Predicted acc  Predicted good  Predicted vgood
Actual unacc              140              4               0                0
Actual acc                  0             36               0                0
Actual good                 0             46               0                0
Actual vgood                0             34               0                0

XGBoost Confusion Matrix (Test):
              Predicted unacc  Predicted acc  Predicted good  Predicted vgood
Actual unacc              144              0               0                0
Actual acc                 24             12               0                0
Actual good                 0             46               0                0
Actual vgood                0             34               0                0

Final Comparison — Decision Tree vs. XGBoost (Validation & Test):


,Model,Split,Accuracy,Avg Precision,Avg Recall,Avg F-score
0,Decision Tree,Validation,0.756757,0.345843,0.413807,0.374717
1,Decision Tree,Test,0.676923,0.325000,0.493056,0.361863
2,XGBoost,Validation,0.691120,0.525388,0.344986,0.344050
3,XGBoost,Test,0.600000,0.246894,0.333333,0.277644
